# Pillar 1 - Phase 1 baseline (no-train)

See `plans/PLAN.md` (Tru cot 1) and `phases/phase-1-no-train.md`.

Runs on Kaggle CPU (no GPU needed - sequences are short). Covers:
- **1.1a-d**: where does the current Vietnamese BPE tokenizer fail (rare compounds, proper nouns, numbers, typos)
- **1.2a-b**: does next-byte entropy peak align with syllable boundaries, compared across a multilingual model and a Vietnamese-pretrained model

Vietnamese is analytic: each whitespace-separated unit is already one syllable, so
whitespace positions are used as ground-truth syllable boundaries.


In [ ]:
import torch, json, statistics, re
from transformers import AutoTokenizer, AutoModelForCausalLM
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# 1.1: word lists
common_words = [
    "con người", "học sinh", "gia đình", "công việc", "thời gian", "buổi sáng",
    "cuộc sống", "quê hương", "bạn bè", "trường học", "thành phố", "đất nước",
    "sức khỏe", "niềm vui", "câu chuyện", "con đường", "bữa cơm", "giấc ngủ",
]

rare_compounds = [
    "khấp khiễng", "khệnh khạng", "lẩn khuất", "chông chênh", "khúm núm",
    "ngúng nguẩy", "rón rén", "khù khờ", "lù đù", "ngơ ngác", "chênh vênh",
    "lấp lửng", "ngổn ngang", "khắc khoải", "dùng dằng", "thất thểu",
    "ngật ngưỡng", "chán chường", "khấp khởi", "lởn vởn",
]

proper_nouns = [
    "Ea H'leo", "Mường Lát", "Pác Nặm", "Bát Xát", "Na Hang", "Trùng Khánh",
    "Xín Mần", "Mèo Vạc", "Đắk Glei", "Krông Pắk", "Nậm Pồ", "Mường Nhé",
    "Nguyễn Thị Thanh Vân", "Đinh Xuân Bá", "Vừ A Sính", "Hoàng Thị Kim Chi",
    "Lò Văn Sang", "Giàng Seo Phử",
]

numbers = [
    "12/09/2026", "1.234.567 VNĐ", "$99.99", "MST: 0312345678-001",
    "0987654321", "23:59:59 ngày 31/12/2025", "Điều 12, Khoản 3",
    "3.14159265", "100.000.000 đồng", "GD-2026-00123", "+84 912 345 678",
    "50%/năm", "km số 27+500",
]

clean_text = (
    "Hôm nay trời đẹp, chúng tôi cùng nhau đi dạo quanh hồ và trò chuyện về "
    "những dự định trong tương lai. Ai cũng cảm thấy vui vẻ và tràn đầy hy vọng."
)
noisy_text = (
    "Hom nay troi dep, chung toi cung nhau di dao quanh ho va tro chuyen ve "
    "nhung du dinh trong tuong lai. Ai cung cam thay vui ve va tran day hy vong."
)


In [ ]:
# NlpHUST/gpt2-vietnamese instead of vinai/PhoGPT-4B: PhoGPT's MPT config raises
# StrictDataclassFieldValidationError on current transformers (attn_pdrop int vs float).
bpe_tok = AutoTokenizer.from_pretrained("NlpHUST/gpt2-vietnamese")

def syllables(s):
    return len(s.split())

def tokens(s, tok):
    return len(tok.encode(s, add_special_tokens=False))

def ratio_for_list(words, tok):
    ratios = [tokens(w, tok) / max(syllables(w), 1) for w in words]
    return statistics.mean(ratios)

baseline_ratio = ratio_for_list(common_words, bpe_tok)
results_1_1 = {}
for name, wl in [("1_1a_rare_compounds", rare_compounds),
                 ("1_1b_proper_nouns", proper_nouns),
                 ("1_1c_numbers", numbers)]:
    r = ratio_for_list(wl, bpe_tok)
    results_1_1[name] = {
        "avg_token_per_syllable": r,
        "baseline_avg_token_per_syllable": baseline_ratio,
        "multiplier_vs_baseline": r / baseline_ratio,
        "pass_threshold_2x": (r / baseline_ratio) > 2.0,
    }

clean_n = tokens(clean_text, bpe_tok)
noisy_n = tokens(noisy_text, bpe_tok)
results_1_1["1_1d_typos"] = {
    "clean_tokens": clean_n,
    "noisy_tokens": noisy_n,
    "multiplier_vs_baseline": noisy_n / clean_n,
    "pass_threshold_2x": (noisy_n / clean_n) > 2.0,
}

for k, v in results_1_1.items():
    print(k, "->", v)


In [ ]:
# 1.2: entropy peak vs syllable boundary
sample_text = (
    "Việt Nam là một quốc gia nằm ở khu vực Đông Nam Á, có bờ biển dài và "
    "nền văn hóa lâu đời. Người dân nơi đây rất cần cù và hiếu khách."
)

def syllable_boundary_positions(text):
    positions = {m.end() for m in re.finditer(r"\s", text)}
    positions.add(0)
    return positions

gt_boundaries = syllable_boundary_positions(sample_text)

@torch.no_grad()
def entropy_peak_overlap(model_name, text, gt_boundaries, top_k_frac=0.3):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    enc = tok(text, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    logits = model(**enc).logits[0]
    probs = torch.softmax(logits.float(), dim=-1)
    entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum(-1)
    entropy = entropy.cpu().tolist()

    char_positions = [end for (_, end) in offsets]
    n_peaks = max(int(len(entropy) * top_k_frac), 1)
    peak_idx = sorted(range(len(entropy)), key=lambda i: entropy[i], reverse=True)[:n_peaks]
    peak_char_positions = {char_positions[i] for i in peak_idx}

    hit = sum(1 for p in peak_char_positions if any(abs(p - b) <= 1 for b in gt_boundaries))
    overlap_pct = hit / max(len(peak_char_positions), 1) * 100

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return overlap_pct

results_1_2 = {}
results_1_2["1_2a_xglm564M"] = entropy_peak_overlap("facebook/xglm-564M", sample_text, gt_boundaries)
results_1_2["1_2b_gpt2_vietnamese"] = entropy_peak_overlap("NlpHUST/gpt2-vietnamese", sample_text, gt_boundaries)

for k, v in results_1_2.items():
    print(k, "-> overlap_pct:", round(v, 1), "pass_threshold_70pct:", v > 70)


In [ ]:
import os
out_dir = "/kaggle/working"
os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, "metrics_pillar1_phase1.jsonl"), "w") as f:
    f.write(json.dumps({"section": "1.1", "results": results_1_1}) + "\n")
    f.write(json.dumps({"section": "1.2", "results": results_1_2}) + "\n")
print("wrote", os.path.join(out_dir, "metrics_pillar1_phase1.jsonl"))
